<a href="https://colab.research.google.com/github/Zikai-Dou/dataflow-ai-labs/blob/main/Lab5_Debugging_Compilation_Stack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 5: Debugging the Compilation Stack

**Course**: *Dataflow Architectures for AI: From Principles to Practice*  
**Module**: 5, Debugging Across the Dataflow Stack  
**Instructor**: Dr. Kaoutar El Maghraoui (IBM Research)  
**Runtime**: Google Colab, GPU runtime required (T4)  
**Expected time**: 90–120 minutes


---

## Learning objectives
By the end of this lab you will be able to:
1. Detect and fix graph breaks with torch._dynamo.explain() and TORCH_LOGS.
2. Quantify numerical divergence between eager and compiled execution.
3. Apply bisection across compilation stages to isolate where a numerical change enters.
4. Classify failures into the diagnostic-funnel layers from lecture.

## Prerequisites
Labs 1-4 complete, plus the Module 5 lecture on debugging across the stack, bisection, and the diagnostic funnel.

## Working through this lab
1. Run the notebook cells in order.
2. Complete each learner task before opening its **Hint**.
3. If you get stuck, expand the **Hint** to view the full reference solution from the authoring notebook.
4. Run each **Self-check** cell after completing the corresponding task.



## Environment Preflight

This cell detects your runtime environment and sets the execution mode:
- **Full GPU**: all exercises run on CUDA (recommended: T4 or better in Colab)
- **Reduced CPU**: GPU-dependent cells use CPU fallback with reduced workloads
- **Static read-only**: no execution; pre-computed outputs are displayed

In [ ]:
# === Environment Preflight ===
import sys
import importlib

# Check Python version
assert sys.version_info >= (3, 9), f"Python >= 3.9 required, got {sys.version}"

# Check PyTorch
try:
    import torch
    _torch_version = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
    assert _torch_version >= (2, 2), f"PyTorch >= 2.2 required, got {torch.__version__}"
except ImportError:
    raise RuntimeError("PyTorch not installed. Run: pip install torch>=2.2")

# Detect execution mode
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    MODE = "full-gpu"
    _device_name = torch.cuda.get_device_name(0)
else:
    DEVICE = torch.device("cpu")
    MODE = "reduced-cpu"
    _device_name = "CPU only"

print(f"╔══════════════════════════════════════════════════╗")
print(f"║  Environment Preflight                          ║")
print(f"╠══════════════════════════════════════════════════╣")
print(f"║  Python:    {sys.version.split()[0]:<37s}║")
print(f"║  PyTorch:   {torch.__version__:<37s}║")
print(f"║  Device:    {_device_name:<37s}║")
print(f"║  Mode:      {MODE:<37s}║")
print(f"╚══════════════════════════════════════════════════╝")

if MODE == "reduced-cpu":
    print("\n⚠️  No GPU detected. GPU-dependent cells will use CPU fallback.")
    print("   For full experience: Runtime → Change runtime type → T4 GPU")


---
## 0 · Setup

In [ ]:
!pip -q install transformers==4.44.2

import os
import math
import torch
import torch.nn as nn
import torch._dynamo as dynamo
from torch.profiler import profile, ProfilerActivity, record_function
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(42)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'torch {torch.__version__} | device: {DEV}')
if DEV == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# PyTorch version compatibility check
import torch
print(f'PyTorch version: {torch.__version__}')
assert int(torch.__version__.split('.')[0]) >= 2, (
    f'This lab requires PyTorch 2.x or later. Got {torch.__version__}. '
    f'Run: pip install --upgrade torch'
)
print('✓ PyTorch version compatible')


We use a **deliberately buggy** transformer block that contains common anti-patterns
students encounter when trying to compile real models. Your job is to find and fix them.

In [ ]:
class BuggyTransformerBlock(nn.Module):
    """A transformer block with deliberate compilation anti-patterns.

    This block has THREE bugs that affect compilation:
    1. A graph break (silent performance killer)
    2. A numerical precision issue
    3. A dynamic shape that prevents static compilation
    """
    def __init__(self, hidden_dim=512, num_heads=8, ffn_dim=2048):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads

        self.q_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.o_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)
        self.ffn_up = nn.Linear(hidden_dim, ffn_dim, bias=False)
        self.ffn_down = nn.Linear(ffn_dim, hidden_dim, bias=False)

        # Bug 2 setup: a threshold stored as a Python float that will
        # cause precision-dependent behavior
        self.scale_threshold = 0.5

    def forward(self, x):
        B, S, H = x.shape

        # --- Attention block ---
        h = self.ln1(x)
        q = self.q_proj(h).reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(h).reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(h).reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = torch.softmax(scores, dim=-1)

        # BUG 1: .item() call causes a graph break
        # The student must find this and replace it with a pure-tensor operation
        max_attn = attn.max().item()
        if max_attn > self.scale_threshold:
            attn = attn * 0.9  # "attention dampening" - pedagogical anti-pattern

        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).reshape(B, S, H)
        out = self.o_proj(out)
        x = x + out

        # --- FFN block ---
        h = self.ln2(x)

        # BUG 3: Dynamic shape from Python control flow
        # This creates a graph break when sequence length varies
        if h.shape[1] > 64:
            h = self.ffn_up(h[:, :64, :])  # truncate long sequences
            h = torch.nn.functional.pad(h, (0, 0, 0, S - 64))  # pad back
        else:
            h = self.ffn_up(h)

        # BUG 2: Manual softmax without numerical stability (no max subtraction)
        # This diverges from torch.softmax which subtracts max for stability
        h_exp = torch.exp(h)  # unstable: can overflow in fp16
        h = h_exp / h_exp.sum(dim=-1, keepdim=True)  # manual "softmax" normalization

        h = self.ffn_down(h)
        x = x + h
        return x


# Instantiate
HIDDEN_DIM = 512
NUM_HEADS = 8
FFN_DIM = 2048
BATCH = 2
SEQ = 128

model = BuggyTransformerBlock(HIDDEN_DIM, NUM_HEADS, FFN_DIM).to(DEV).eval()
dummy = torch.randn(BATCH, SEQ, HIDDEN_DIM, device=DEV)

# Verify it runs in eager mode
with torch.no_grad():
    out_eager = model(dummy)
print(f'Eager output shape: {out_eager.shape}')
print(f'Eager output mean: {out_eager.mean().item():.6f}')

---
## 1 · Exercise: Detect Graph Breaks

Use `torch._dynamo.explain()` to detect how many graph breaks exist in the
`BuggyTransformerBlock`. Report the count, locations, and reasons.

Then try compiling with `torch.compile` and measure the performance vs eager
to confirm the graph breaks are hurting performance.

In [ ]:
# 1.1: Detect graph breaks with torch._dynamo.explain()

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# 1.1: Detect graph breaks with torch._dynamo.explain()
dynamo.reset()
expl = dynamo.explain(model)(dummy)          # current API: explain(fn)(*args)
print(f"graph breaks : {expl.graph_break_count}")
print(f"graphs        : {expl.graph_count}")
print("break reasons:")
for i, r in enumerate(expl.break_reasons):
    reason = getattr(r, "reason", r)
    print(f"  [{i+1}] {reason}")
```

</details>


In [ ]:
# 1.2: benchmark eager vs compiled (graph breaks can make 'compiled' no faster)
# Warm-up run to trigger compilation

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# 1.2: benchmark eager vs compiled (graph breaks can make 'compiled' no faster)
from torch.utils import benchmark as torch_bench

def bench_ms(fn, x):
    with torch.no_grad():
        for _ in range(3):
            fn(x)
    if DEV == "cuda":
        torch.cuda.synchronize()
    t = torch_bench.Timer(stmt="with torch.no_grad(): fn(x)",
                          globals={"fn": fn, "x": x, "torch": torch})
    return t.blocked_autorange(min_run_time=1.0).median * 1000

eager_ms    = bench_ms(model, dummy)
model_compiled = torch.compile(model)
# Warm-up run to trigger compilation
_ = model_compiled(dummy)

compiled_ms = bench_ms(model_compiled, dummy)
speedup     = eager_ms / compiled_ms
print(f"eager    : {eager_ms:.3f} ms")
print(f"compiled : {compiled_ms:.3f} ms")
print(f"speedup  : {speedup:.2f}x  (<= ~1x is expected while breaks remain)")
```

</details>


---
## 2 · Exercise: Fix the Graph Breaks

Create a `FixedTransformerBlock` that eliminates all graph breaks while
preserving the intended behavior. The three bugs are:

1. **`.item()` + Python if**: replace with a pure tensor operation (e.g., `torch.where`)
2. **Manual unstable softmax**: replace with `torch.softmax` or add max-subtraction
3. **Shape-dependent branching**: remove the branch or use `torch._dynamo.mark_dynamic`

After fixing, verify with `dynamo.explain()` that break_count == 0.

In [ ]:
        # --- Attention block ---
        # Fix Bug 1: .item() + Python `if`  ->  pure-tensor select (no host sync, no break)
        # --- FFN block ---

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
class FixedTransformerBlock(nn.Module):
    """The same transformer block with all three graph-related bugs fixed."""
    def __init__(self, hidden_dim=512, num_heads=8, ffn_dim=2048):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.q_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.o_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)
        self.ffn_up = nn.Linear(hidden_dim, ffn_dim, bias=False)
        self.ffn_down = nn.Linear(ffn_dim, hidden_dim, bias=False)
        self.scale_threshold = 0.5

    def forward(self, x):
        B, S, H = x.shape

        # --- Attention block ---
        h = self.ln1(x)
        q = self.q_proj(h).reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(h).reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(h).reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = torch.softmax(scores, dim=-1)

        # Fix Bug 1: .item() + Python `if`  ->  pure-tensor select (no host sync, no break)
        attn = torch.where(attn.max() > self.scale_threshold, attn * 0.9, attn)

        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).reshape(B, S, H)
        out = self.o_proj(out)
        x = x + out

        # --- FFN block ---
        h = self.ln2(x)
        # Fix Bug 3: drop the shape-dependent branch; run ffn_up on the full sequence.
        h = self.ffn_up(h)
        # Fix Bug 2: numerically stable torch.softmax instead of manual exp/sum (fp16-safe).
        h = torch.softmax(h, dim=-1)
        h = self.ffn_down(h)
        x = x + h
        return x


# Instantiate fixed model with same weights
model_fixed = FixedTransformerBlock(HIDDEN_DIM, NUM_HEADS, FFN_DIM).to(DEV).eval()
model_fixed.load_state_dict(model.state_dict())

# Verify no graph breaks
dynamo.reset()
expl_fixed = dynamo.explain(model_fixed)(dummy)
print(f"fixed model graph breaks: {expl_fixed.graph_break_count}")
assert expl_fixed.graph_break_count == 0, "Fixed model should have zero graph breaks"
print("[OK] zero graph breaks")
```

</details>


In [ ]:
# Warm-up compiled model (triggers compilation)

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# 2.3: compile the fixed model and generate reference outputs
dynamo.reset()
model_fixed_compiled = torch.compile(model_fixed)

# Warm-up compiled model (triggers compilation)
with torch.no_grad():
    _ = model_fixed_compiled(dummy)

# Generate all reference outputs needed for numerical comparison
with torch.no_grad():
    out_buggy_eager   = model(dummy)           # buggy model, eager
    out_fixed_eager   = model_fixed(dummy)     # fixed model, eager
    out_fixed_compiled = model_fixed_compiled(dummy)  # fixed model, compiled

print(f"model_fixed_compiled ready on {DEV}")
print(f"out_buggy_eager   shape: {out_buggy_eager.shape}")
print(f"out_fixed_eager   shape: {out_fixed_eager.shape}")
print(f"out_fixed_compiled shape: {out_fixed_compiled.shape}")
```

</details>


In [ ]:
# 2.4: benchmark fixed compiled vs buggy compiled

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# 2.4: benchmark fixed compiled vs buggy compiled
fixed_compiled_ms = bench_ms(model_fixed_compiled, dummy)
print(f"buggy compiled : {compiled_ms:.3f} ms")
print(f"fixed compiled : {fixed_compiled_ms:.3f} ms")
print(f"speedup from removing breaks: {compiled_ms / fixed_compiled_ms:.2f}x")
```

</details>


---
## 3 · Exercise: Numerical Divergence Analysis

Now compare the **eager** output of the buggy model against the **eager** output of the
fixed model. The fixes you applied changed the numerical behavior (especially Bug 2: the
unstable softmax). Quantify the divergence.

Then compare the fixed model in **eager** vs **compiled** mode to see what level of
divergence is "normal" from compilation alone.

In [ ]:
# 3.1: buggy eager vs fixed eager

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# 3.1: buggy eager vs fixed eager
diff = (out_buggy_eager - out_fixed_eager).abs()
max_abs_diff  = diff.max().item()
mean_abs_diff = diff.mean().item()
max_rel_diff  = (diff / (out_fixed_eager.abs() + 1e-8)).max().item()
print(f"max abs diff : {max_abs_diff:.6e}")
print(f"mean abs diff: {mean_abs_diff:.6e}")
print(f"max rel diff : {max_rel_diff:.6e}")
print("Note: this gap is dominated by Bug 3 (the FFN was truncated to 64 positions and")
print("zero-padded). In fp32, Bug 2 (manual vs stable softmax) contributes ~1e-7 only;")
print("Bug 1's dampening branch does not fire for these inputs (attn.max() < 0.5).")
```

</details>


In [ ]:
# 3.2: fixed eager vs fixed compiled (divergence from compilation alone)

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# 3.2: fixed eager vs fixed compiled (divergence from compilation alone)
d2 = (out_fixed_eager - out_fixed_compiled).abs()
compilation_max_abs  = d2.max().item()
compilation_mean_abs = d2.mean().item()
compilation_max_rel  = (d2 / (out_fixed_eager.abs() + 1e-8)).max().item()
print(f"compilation max abs : {compilation_max_abs:.6e}")
print(f"compilation mean abs: {compilation_mean_abs:.6e}")
print(f"compilation max rel : {compilation_max_rel:.6e}")
```

</details>


In [ ]:
    # The model-level fix changes results far more than compilation noise does:

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
comparison = {
    "buggy_vs_fixed_max_abs":    max_abs_diff,
    "buggy_vs_fixed_mean_abs":   mean_abs_diff,
    "compilation_only_max_abs":  compilation_max_abs,
    "compilation_only_mean_abs": compilation_mean_abs,
    # The model-level fix changes results far more than compilation noise does:
    "is_bug2_divergence_larger": max_abs_diff > compilation_max_abs,
    "ratio":                     (max_abs_diff / compilation_max_abs) if compilation_max_abs > 0 else float("inf"),
}

print("Divergence comparison:")
for k, v in comparison.items():
    print(f"  {k}: {v}")
```

</details>


### Tolerance Selection and False Positives

Choosing `atol` and `rtol` requires understanding:
1. **Dtype precision**: FP32 has ~7 decimal digits; FP16 has ~3.3; BF16 has ~3.3 but with FP32 range
2. **Accumulation order**: reductions (sum, mean, softmax) are non-associative in floating point
3. **Determinism**: some CUDA ops are non-deterministic by default (atomics in backward pass)

| Comparison | Typical `atol` | Typical `rtol` | Rationale |
|---|---|---|---|
| eager vs eager (same seed) | 0 | 0 | Bitwise identical |
| eager vs compiled (FP32) | 1e-5 | 1e-5 | Reordered ops |
| eager vs compiled (FP16) | 1e-2 | 1e-2 | Low precision + reordering |
| across GPU architectures | 1e-3 | 1e-3 | Different rounding hardware |

> ⚠️ **False positive risk:** Setting `atol` too tight flags benign reordering as bugs.
> Setting it too loose hides real numerical issues.

In [ ]:
# Demonstrate false positives from overly tight tolerance
torch.manual_seed(42)
x = torch.randn(128, 768, device=DEVICE if 'DEVICE' in dir() else 'cpu')

# Simulate accumulation-order difference (sum in different order)
sum_fwd = x.sum()
sum_rev = x.flip(0).sum()  # same values, different addition order

diff = (sum_fwd - sum_rev).abs().item()
print(f"Same data, different sum order:")
print(f"  |sum(x) - sum(flip(x))| = {diff:.2e}")
print(f"  Would torch.allclose(atol=1e-7) pass? {torch.allclose(sum_fwd, sum_rev, atol=1e-7)}")
print(f"  Would torch.allclose(atol=1e-4) pass? {torch.allclose(sum_fwd, sum_rev, atol=1e-4)}")

# Best practice: scale tolerance by dtype
def recommended_tolerance(dtype):
    """Return (atol, rtol) appropriate for comparing compiled vs eager outputs."""
    if dtype == torch.float32:
        return 1e-5, 1e-5
    elif dtype == torch.float16:
        return 1e-2, 1e-2
    elif dtype == torch.bfloat16:
        return 1e-2, 1e-2
    else:
        return 1e-5, 1e-5

atol, rtol = recommended_tolerance(x.dtype)
print(f"\nRecommended for {x.dtype}: atol={atol}, rtol={rtol}")
print(f"  torch.allclose passes: {torch.allclose(sum_fwd, sum_rev, atol=atol, rtol=rtol)}")

---
## 4 · Exercise: Bisection Debugging

Now practice the **bisection method** from lecture. We'll introduce a deliberate
correctness bug in a compilation backend and use bisection to find which stage broke.

The setup: we compile the fixed model with different backends and compare outputs
at each stage to narrow down where divergence exceeds the expected BF16 noise level.

In [ ]:
# Bisection setup: compare outputs at each compilation stage
# Stage 0: Eager baseline (ground truth)
# Stage 1: Graph capture only (no optimization) using "eager" backend

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# Bisection setup: compare outputs at each compilation stage

dynamo.reset()

# Stage 0: Eager baseline (ground truth)
with torch.no_grad():
    baseline = model_fixed(dummy)

# Stage 1: Graph capture only (no optimization) using "eager" backend
dynamo.reset()
model_stage1 = torch.compile(model_fixed, backend="eager")
with torch.no_grad():
    out_stage1 = model_stage1(dummy)

# Stage 2: AOT autograd decomposition (no codegen) using "aot_eager" backend
dynamo.reset()
model_stage2 = torch.compile(model_fixed, backend="aot_eager")
with torch.no_grad():
    out_stage2 = model_stage2(dummy)

# Stage 3: Full compilation with Inductor backend
dynamo.reset()
model_stage3 = torch.compile(model_fixed, backend="inductor")
with torch.no_grad():
    out_stage3 = model_stage3(dummy)

print("Bisection results (max abs diff from eager baseline):")
print(f"  Stage 1 (graph capture only):  {(baseline - out_stage1).abs().max().item():.2e}")
print(f"  Stage 2 (AOT decomposition):   {(baseline - out_stage2).abs().max().item():.2e}")
print(f"  Stage 3 (full Inductor):       {(baseline - out_stage3).abs().max().item():.2e}")
```

</details>


In [ ]:
    # eager / aot_eager reproduce eager numerics (~0); Inductor is the first stage
    # whose fused / reordered reductions move the result above fp32 noise.

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
bisection_result = {
    # eager / aot_eager reproduce eager numerics (~0); Inductor is the first stage
    # whose fused / reordered reductions move the result above fp32 noise.
    "first_divergent_stage": 3,
    "divergence_cause": ("TorchInductor fuses elementwise ops and reorders reduction "
                         "accumulation, so floating-point results differ from eager at "
                         "the ~1e-4-1e-3 level; graph capture (eager) and AOT decomposition "
                         "(aot_eager) do not change the math."),
    "is_this_a_bug": False,   # expected floating-point accumulation difference, not a correctness bug
}

print("Bisection interpretation:")
for k, v in bisection_result.items():
    print(f"  {k}: {v}")
```

</details>


---
## 5 · Exercise: TORCH_COMPILE_DEBUG Inspection

Use `TORCH_COMPILE_DEBUG` to dump the compilation artifacts and inspect what the
compiler actually did to your fixed model.

In [ ]:
# Run compilation with debug output
# Clean up env var
# List what was produced

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# Run compilation with debug output
import shutil
debug_dir = '/tmp/torch_compile_debug'
if os.path.exists(debug_dir):
    shutil.rmtree(debug_dir)

os.environ['TORCH_COMPILE_DEBUG'] = '1'
os.environ['TORCH_COMPILE_DEBUG_DIR'] = debug_dir

dynamo.reset()
model_debug = torch.compile(model_fixed, backend="inductor")
with torch.no_grad():
    _ = model_debug(dummy)

# Clean up env var
del os.environ['TORCH_COMPILE_DEBUG']
del os.environ['TORCH_COMPILE_DEBUG_DIR']

# List what was produced
import glob
artifacts = glob.glob(f'{debug_dir}/**/*', recursive=True)
print(f"Debug artifacts produced: {len(artifacts)} files")
for a in sorted(artifacts)[:20]:
    print(f"  {a.replace(debug_dir, '.')}")
```

</details>


In [ ]:
# 5.1: find and read the captured FX graph

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# 5.1: find and read the captured FX graph
import glob
fx_files = (glob.glob(f"{debug_dir}/**/*fx_graph_readable*", recursive=True)
            or glob.glob(f"{debug_dir}/**/*readable*", recursive=True))
if fx_files:
    txt = open(fx_files[0]).read()
    print(f"FX graph: {fx_files[0]}\n" + "=" * 60)
    print("\n".join(txt.splitlines()[:60]))
    n_ops    = txt.count(" = torch.ops.")
    n_matmul = txt.count("aten.mm") + txt.count("aten.bmm") + txt.count("aten.addmm")
    print("=" * 60)
    print(f"approx operators in graph: {n_ops} | matmul-family ops: {n_matmul}")
else:
    print("No fx_graph_readable file found. Re-run the TORCH_COMPILE_DEBUG cell with dynamo.reset() first.")
```

</details>


In [ ]:
# 5.2: read the generated kernels and count Triton vs extern

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# 5.2: read the generated kernels and count Triton vs extern
oc = glob.glob(f"{debug_dir}/**/output_code.py", recursive=True)
if oc:
    code = open(oc[0]).read()
    triton = code.count("def triton_")
    extern = code.count("extern_kernels.")
    print(f"output_code: {oc[0]}")
    print(f"Triton (fused) kernels : {triton}")
    print(f"extern (cuBLAS) kernels: {extern}")
    print("Fused = elementwise + norm groups; extern = the GEMMs (q/k/v/o + ffn).")
else:
    print("No output_code.py found. Ensure the inductor debug run completed.")
```

</details>


---
## 6 · Exercise: Written Analysis: Diagnostic Funnel Classification

For each of the three bugs you found and fixed, classify it in the **diagnostic funnel**
from lecture. Write **250–400 words** total covering:

1. For each bug: which **layer** of the funnel does it originate at? Which layer would the
   **symptom** manifest at if you were running on a dataflow accelerator (e.g., Spyre)?

2. Which debugging **mindset** (software, compiler, or hardware) was required to find each?

3. The bisection results: what did they tell you about the relationship between compilation
   stages and numerical precision? Is the divergence you observed a "bug" or expected behavior?

4. Predict: if you ran the buggy model on Spyre instead of GPU, which bug would have the
   most **severe** consequence (crash vs. wrong answer vs. perf degradation)? Why?

<details>
<summary><strong>Hint</strong></summary>

**Bug 1:** `.item()` + data-dependent `if` (software → compiler layer). It originates in the *software* layer: a Python-level control decision that reads a tensor value. Its symptom surfaces at the *compiler* layer as a graph break; Dynamo cannot trace across a host sync, so it splits the graph and falls back to eager. Finding it needs the **compiler mindset**: read `dynamo.explain` break reasons rather than staring at the math. On Spyre this is the most severe of the three: a static dataflow compiler cannot emit a host round-trip mid-graph, so instead of a silent GPU perf regression it is likely a **hard compile-time rejection**: the model does not compile until the branch is removed.

**Bug 2:** manual unstable softmax (numerics layer). It originates in the *numerics* layer: `exp(h)/sum(exp(h))` without max-subtraction overflows in fp16. On GPU in fp32 it is nearly invisible (~1e-7 vs `torch.softmax`), so it is a latent correctness risk that only manifests under low precision or large activations; exactly the regime a dataflow accelerator runs in. Finding it needs the **hardware/numerics mindset**: reason about dynamic range, not graph structure. On Spyre the symptom is a **wrong answer** (NaN/Inf propagation) rather than a crash.

**Bug 3:** shape-dependent branch (software → compiler layer). The `if h.shape[1] > 64` truncates and zero-pads, changing the result *and* forcing recompilation/guards on sequence length. It dominates the buggy-vs-fixed divergence measured in Section 3. Compiler mindset again: `TORCH_LOGS=recompiles` or `explain` exposes it.

**Bisection.** Stages 1–2 (eager, aot_eager) match the eager baseline; Stage 3 (Inductor) is the first to exceed fp32 noise. That divergence is **expected**, not a bug: fusion and reduction reordering change accumulation order. The lesson is to set a tolerance from the compiler-only comparison first, then treat anything above it as a real bug. **Most severe on Spyre: Bug 1** (fails to compile), because a scratchpad dataflow target has no eager fallback to hide behind.

</details>


---

## Part 6 - Update the Accelerator Landscape Map

Add this module's row to the shared map you have been building since Lab 1.


In [ ]:
# Part 6 -- Landscape Map row: debuggability of the stack you exercised this module.

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# Part 6 -- Landscape Map row: debuggability of the stack you exercised this module.
landscape_row_w5 = {
    "accelerator":              f"GPU ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "CPU",
    "debuggability":            "eager fallback hides graph breaks; intermediate values inspectable",
    "graph_breaks_found":       int(expl.graph_break_count),
    "compilation_noise_maxabs": round(compilation_max_abs, 8),
    "first_divergent_stage":    bisection_result["first_divergent_stage"],
    "spyre_prediction":         "no eager fallback: a graph break becomes a compile-time rejection",
}
for _k, _v in landscape_row_w5.items():
    print(f"  {_k:26s}: {_v}")
```

</details>


---
## ★ Optional: Dataflow Deep Dive (Spyre hardware)
_Requires Spyre access via an IBM Research partnership. Skip on the core track._

1. Attempt to compile the `BuggyTransformerBlock` on Spyre (`torch_sendnn`).
2. Document which bugs become **compilation errors** (Spyre rejects) vs **runtime errors**.
3. Use `SENDNN_LOG_LEVEL=DEBUG` to capture the Spyre compiler's IR and identify where
   each bug manifests in the Spyre compilation pipeline.
4. Compare: on GPU, Bug 1 is a silent perf regression. On Spyre, does it compile at all?

> **Spyre vocabulary (consistent across all labs):** `import torch_sendnn` registers the `aiu` device; compile with `torch.compile(model, backend="sendnn")`; dump the backend IR with `TORCH_COMPILE_DEBUG=1` (look for `sdsc_*.json`); raise verbosity with `SENDNN_LOG_LEVEL=DEBUG`. Requires Spyre access via an IBM Research partnership.


In [ ]:
# Spyre track only, pseudocode skeleton
# import torch_sendnn
# model_spyre = BuggyTransformerBlock(HIDDEN_DIM, NUM_HEADS, FFN_DIM).to('aiu').eval()
# try:
#     compiled_spyre = torch.compile(model_spyre, backend='sendnn')
#     out = compiled_spyre(dummy.to('aiu'))
# except Exception as e:
#     print(f"Spyre compilation failed: {e}")
#     # Which bug caused this? Document it.